In [ ]:
import pandas as pd
from transformers import pipeline

In [ ]:
df = pd.read_excel("news.xlsx")
news_list = df["News"].tolist()

In [ ]:
news_list

['Prince Harry and his wife Meghan, Duchess of Sussex, both revealed more of the unhappiness they experienced as working royals on Thursday, with Harry saying he didn’t want this job after his mother died and Meghan claiming she was the most trolled person in the entire world. Harry has spoken extensively about the devastating impact of his mother’s death, citing it as the reason for his protectiveness over his wife and young family.',
 'The South African leftwing politician Julius Malema has been sentenced to five years in prison for firing a rifle in the air at a political rally in 2018.',
 'Three people were killed in a US strike on another alleged drug-trafficking boat, the fifth such deadly attack in as many days, military officials have announced.',
 'But the trip, and a meeting with his Chinese counterpart, Xi Jinping, was kicked back by several weeks after Trump decided to launch strikes with Israel against Iran, starting a war in the Middle East that has caused a global energy

In [ ]:
#pip install "transformers<5"


Суммаризация



In [ ]:
sum_bart = pipeline("text2text-generation", model="facebook/bart-large-cnn")

Device set to use cuda:0


In [ ]:
sum_pegasus = pipeline("text2text-generation", model="google/pegasus-xsum")


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


In [ ]:
def sum(pipe, text, max_len=50, min_len=20):
    return pipe(text, max_length=max_len, min_length=min_len, do_sample=False)[0]['generated_text']

df["Summary_BART"] = [sum(sum_bart, t) for t in news_list]
df["Summary_Pegasus"] = [sum(sum_pegasus, t) for t in news_list]

df.head()

,News,Summary_BART,Summary_Pegasus
0,"Prince Harry and his wife Meghan, Duchess of S...","Prince Harry and his wife Meghan, Duchess of S...","Prince Harry and his wife Meghan, Duchess of S..."
1,The South African leftwing politician Julius M...,Julius Malema sentenced to five years in priso...,"The BBC's Mark Lowen reports from the capital,..."
2,Three people were killed in a US strike on ano...,Three people were killed in a US strike on ano...,The US has carried out a series of air strikes...
3,"But the trip, and a meeting with his Chinese c...",The trip was kicked back by several weeks afte...,President Donald Trump's first foreign trip si...
4,"Australia’s two biggest car-share companies, G...",GoGet and Flexicar have removed fuel cards fro...,Car-share companies in Australia have been hit...


Классификация

In [ ]:
classifier_modern = pipeline("text-classification", model="Sengil/ModernBERT-NewsClassifier-EN-small")
df["Class_ModernBERT"] = [classifier_modern(t)[0]['label'] for t in news_list]

Device set to use cuda:0


In [ ]:
classifier_zero = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
categories = ["politics", "sports", "technology", "economy", "culture", "science"]
df["Class_ZeroShot"] = [classifier_zero(t, categories)['labels'][0] for t in news_list]

Device set to use cuda:0


In [ ]:
df

,News,Summary_BART,Summary_Pegasus,Class_ModernBERT,Class_ZeroShot
0,"Prince Harry and his wife Meghan, Duchess of S...","Prince Harry and his wife Meghan, Duchess of S...","Prince Harry and his wife Meghan, Duchess of S...",ENTERTAINMENT,culture
1,The South African leftwing politician Julius M...,Julius Malema sentenced to five years in priso...,"The BBC's Mark Lowen reports from the capital,...",BLACK VOICES,politics
2,Three people were killed in a US strike on ano...,Three people were killed in a US strike on ano...,The US has carried out a series of air strikes...,POLITICS,technology
3,"But the trip, and a meeting with his Chinese c...",The trip was kicked back by several weeks afte...,President Donald Trump's first foreign trip si...,POLITICS,politics
4,"Australia’s two biggest car-share companies, G...",GoGet and Flexicar have removed fuel cards fro...,Car-share companies in Australia have been hit...,BUSINESS,technology
5,A last-ditch effort to rescue a wayward whale ...,The 12-tonne whale has transfixed Germans for ...,"Footage courtesy of AFP, AP, EPA, Getty Images...",POLITICS,culture
6,More than 100 writers have quit the historic F...,More than 100 writers have quit the historic F...,France's literary world has been rocked by one...,BUSINESS,culture
7,"Bhosle, who recorded more than 12,000 songs, b...","Bhosle recorded more than 12,000 songs. She be...",Legendary Indian playback singer Asha Bhosle h...,ENTERTAINMENT,sports
8,Senate Democrats move to stall Trump’s ‘absurd...,Democratic lawmakers on the Senate banking com...,US President Donald Trump’s pick to be the nex...,POLITICS,politics
9,Young men are now more likely than young women...,Young men are now more likely than young women...,Young men are now more likely than young women...,BUSINESS,science


люди и локации

In [ ]:
ner_dslim = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

def extract_people_locations(pipe, text):
    ents = pipe(text)
    people = [e['word'] for e in ents if e['entity_group'] == 'PER']
    locs = [e['word'] for e in ents if e['entity_group'] == 'LOC']
    return ', '.join(people), ', '.join(locs)

people1, locs1 = zip(*[extract_people_locations(ner_dslim, t) for t in news_list])
df["NER_dslim_People"] = people1
df["NER_dslim_Locations"] = locs1

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [ ]:
ner_multi = pipeline("ner", model="Davlan/distilbert-base-multilingual-cased-ner-hrl", aggregation_strategy="simple")
people2, locs2 = zip(*[extract_people_locations(ner_multi, t) for t in news_list])
df["NER_multi_People"] = people2
df["NER_multi_Locations"] = locs2

Device set to use cuda:0


In [ ]:
df

,News,Summary_BART,Summary_Pegasus,Class_ModernBERT,Class_ZeroShot,NER_dslim_People,NER_dslim_Locations,NER_multi_People,NER_multi_Locations
0,"Prince Harry and his wife Meghan, Duchess of S...","Prince Harry and his wife Meghan, Duchess of S...","Prince Harry and his wife Meghan, Duchess of S...",ENTERTAINMENT,culture,"Harry, Meg, ##han, Harry, Meg, ##han, Harry",Sussex,"Harry, Meghan, Duchess of Sussex, Harry, Megha...",
1,The South African leftwing politician Julius M...,Julius Malema sentenced to five years in priso...,"The BBC's Mark Lowen reports from the capital,...",BLACK VOICES,politics,Julius Malema,,Julius Malema,
2,Three people were killed in a US strike on ano...,Three people were killed in a US strike on ano...,The US has carried out a series of air strikes...,POLITICS,technology,,US,,US
3,"But the trip, and a meeting with his Chinese c...",The trip was kicked back by several weeks afte...,President Donald Trump's first foreign trip si...,POLITICS,politics,"Xi Jinping, Trump","Israel, Iran, Middle East","Xi Jinping, Trump","Israel, Iran, Middle East"
4,"Australia’s two biggest car-share companies, G...",GoGet and Flexicar have removed fuel cards fro...,Car-share companies in Australia have been hit...,BUSINESS,technology,,"Australia, Melbourne",,"Australia, Melbourne"
5,A last-ditch effort to rescue a wayward whale ...,The 12-tonne whale has transfixed Germans for ...,"Footage courtesy of AFP, AP, EPA, Getty Images...",POLITICS,culture,,Baltic Sea,,Baltic Sea
6,More than 100 writers have quit the historic F...,More than 100 writers have quit the historic F...,France's literary world has been rocked by one...,BUSINESS,culture,,,,
7,"Bhosle, who recorded more than 12,000 songs, b...","Bhosle recorded more than 12,000 songs. She be...",Legendary Indian playback singer Asha Bhosle h...,ENTERTAINMENT,sports,"B, ##hosle",,Bhosle,
8,Senate Democrats move to stall Trump’s ‘absurd...,Democratic lawmakers on the Senate banking com...,US President Donald Trump’s pick to be the nex...,POLITICS,politics,"Trump, Kevin Wars, Kevin Warsh, Trump, Jerome ...",,"Trump, Kevin Warsh, Kevin Warsh, Trump, Jerome...",
9,Young men are now more likely than young women...,Young men are now more likely than young women...,Young men are now more likely than young women...,BUSINESS,science,,US,Gallup,US


Метрики - без разметки

Суммаризация — BERTScore (семантическое сходство с оригиналом)

In [ ]:
!pip install bert-score

from bert_score import BERTScorer
scorer = BERTScorer(lang="en", rescale_with_baseline=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.0 MB/s eta 0:00:00


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
bart_f1 = []
pegasus_f1 = []
for idx in range(len(df)):
    orig = df.loc[idx, 'News']
    bart = df.loc[idx, 'Summary_BART']
    peg = df.loc[idx, 'Summary_Pegasus']
    _, _, f1_b = scorer.score([bart], [orig])
    _, _, f1_p = scorer.score([peg], [orig])
    bart_f1.append(f1_b.item())
    pegasus_f1.append(f1_p.item())


In [ ]:
df['BERTScore_BART'] = bart_f1
df['BERTScore_Pegasus'] = pegasus_f1

print("Средний BERTScore BART:", df['BERTScore_BART'].mean())
print("Средний BERTScore PEGASUS:", df['BERTScore_Pegasus'].mean())

Средний BERTScore BART: 0.7295802175998688
Средний BERTScore PEGASUS: 0.28394374027848246


Классификация — согласованность двух моделей (cohen_kappa_score)

In [ ]:
from sklearn.metrics import cohen_kappa_score

df['Class_ZeroShot_upper'] = df['Class_ZeroShot'].str.upper()
agreement = (df['Class_ModernBERT'] == df['Class_ZeroShot_upper']).mean()
print(f"Согласие после приведения регистра: {agreement:.2%}")

Согласие после приведения регистра: 20.00%


Люди и локации: NER — Jaccard similarity  (между двумя моделями)

In [ ]:
def jaccard_sim(str1, str2):
    set1 = set([x.strip() for x in str(str1).split(',') if x.strip()])
    set2 = set([x.strip() for x in str(str2).split(',') if x.strip()])
    if not set1 and not set2:
        return 1.0
    inter = len(set1 & set2)
    union = len(set1 | set2)
    return inter / union if union > 0 else 0.0

In [ ]:
people_jacc = []
loc_jacc = []
for idx in range(len(df)):
    p1 = df.loc[idx, 'NER_dslim_People']
    p2 = df.loc[idx, 'NER_multi_People']
    l1 = df.loc[idx, 'NER_dslim_Locations']
    l2 = df.loc[idx, 'NER_multi_Locations']
    people_jacc.append(jaccard_sim(p1, p2))
    loc_jacc.append(jaccard_sim(l1, l2))

In [ ]:
df['NER_People_Agreement'] = people_jacc
df['NER_Locations_Agreement'] = loc_jacc

print("Средний Jaccard (People):", df['NER_People_Agreement'].mean())
print("Средний Jaccard (Locations):", df['NER_Locations_Agreement'].mean())


Средний Jaccard (People): 0.6950000000000001
Средний Jaccard (Locations): 0.9


реализация Attention (не без помощи ИИ)

In [ ]:
pip install torch


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [19]:
class Attention(nn.Module):
    def forward(self, Q, K, V, mask=None):
        # Q, K, V: (batch, seq_len, d_model)
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)  # (batch, seq_q, seq_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)

In [20]:
class Encoder(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.attn = Attention()
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        a = self.attn(x, x, x, mask)
        x = self.norm1(x + a)
        f = self.ff(x)
        x = self.norm2(x + f)
        return x

In [21]:
class Decoder(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.self_attn = Attention()
        self.cross_attn = Attention()
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        a1 = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + a1)
        a2 = self.cross_attn(x, enc_out, enc_out, src_mask)
        x = self.norm2(x + a2)
        f = self.ff(x)
        x = self.norm3(x + f)
        return x

In [23]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=32, d_ff=64):
        super().__init__()
        self.enc_emb = nn.Embedding(vocab_size, d_model)
        self.dec_emb = nn.Embedding(vocab_size, d_model)
        self.encoder = Encoder(d_model, d_ff)
        self.decoder = Decoder(d_model, d_ff)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(2)
        sz = tgt.size(1)
        future_mask = torch.tril(torch.ones(sz, sz)).bool().to(tgt.device)
        tgt_mask = tgt_mask & future_mask.unsqueeze(0).unsqueeze(0)
        src_mask = src_mask.squeeze(1)
        tgt_mask = tgt_mask.squeeze(1)

        # Энкодер
        e = self.enc_emb(src)
        e = self.encoder(e, src_mask)

        # Декодер
        d = self.dec_emb(tgt)
        d = self.decoder(d, e, src_mask, tgt_mask)

        # Выход
        logits = self.out(d)
        return logits


In [24]:
train_data = [
    torch.tensor([1, 2, 3]),
    torch.tensor([2, 3, 4, 5]),
    torch.tensor([5, 1, 2]),
    torch.tensor([1, 1, 2, 2])
]

In [25]:
vocab_size = 6
model = Transformer(vocab_size, d_model=16, d_ff=32)
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss(ignore_index=0)


In [26]:
def pad_batch(batch):
    max_len = max(len(x) for x in batch)
    return torch.stack([torch.cat([x, torch.zeros(max_len - len(x), dtype=torch.long)]) for x in batch])

In [28]:
for epoch in range(200):
    total_loss = 0
    for i in range(0, len(train_data), 2):
        src = pad_batch(train_data[i:i+2])
        tgt = pad_batch([s.clone() for s in train_data[i:i+2]])

        logits = model(src, tgt[:, :-1])
        loss = criterion(logits.reshape(-1, vocab_size), tgt[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} loss = {total_loss:.4f}")

Epoch   0 loss = 0.2799
Epoch  50 loss = 0.2791
Epoch 100 loss = 0.2786
Epoch 150 loss = 0.2785


In [29]:
model.eval()
test_src = torch.tensor([[1, 2, 3]])
with torch.no_grad():
    start = test_src[:, :-1]  # [1,2]
    out = model(test_src, start)
    pred = out.argmax(-1)
print("\nТест:")
print(f"Вход:          {test_src[0].tolist()}")
print(f"Предсказание:  {pred[0].tolist()}")
print(f"Должно быть:   {test_src[0, 1:].tolist()}")


Тест:
Вход:          [1, 2, 3]
Предсказание:  [2, 3]
Должно быть:   [2, 3]
